<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/advanced/06_inference_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inference Optimization

> **Track:** ML Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
A model that takes 2 seconds per request is useless at scale. This notebook covers a **systematic toolkit** for reducing model inference latency and increasing throughput: ONNX export, quantization, TorchScript, dynamic batching, and caching.

### What You'll Learn
- Profiling baseline model latency
- ONNX export and the ONNX Runtime
- INT8 and FP16 quantization with PyTorch
- TorchScript compilation
- Dynamic batching for throughput
- Model caching strategies
- Benchmarking methodology

```bash
pip install torch onnx onnxruntime numpy scikit-learn
```

In [ ]:
# Setup: build a baseline model to optimize
import time
import numpy as np
import torch
import torch.nn as nn
from typing import Any
from contextlib import contextmanager

torch.manual_seed(42)

class BaselineClassifier(nn.Module):
    """A moderately-sized classifier for benchmarking optimization techniques."""
    def __init__(self, input_dim: int = 128, hidden_dim: int = 256, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = BaselineClassifier(input_dim=128, hidden_dim=256, num_classes=10)
model.eval()  # always eval mode for inference

# Sample input
sample_input = torch.randn(1, 128)
batch_input = torch.randn(32, 128)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input shape: {sample_input.shape}")

## 1. Profiling Baseline Latency

Always **measure first** before optimizing. A proper benchmark runs many iterations and reports p50/p99 latency — not just the mean.

In [ ]:
# Benchmark latency with proper warmup and p99 measurement

def benchmark_latency(
    model_fn: callable,
    input_tensor: torch.Tensor,
    n_warmup: int = 50,
    n_runs: int = 500,
    label: str = "model"
) -> dict[str, float]:
    """Measure inference latency with warmup. Returns p50/p95/p99 in ms."""
    # Warmup: avoid cold-start JIT effects
    with torch.no_grad():
        for _ in range(n_warmup):
            model_fn(input_tensor)

    # Benchmark
    latencies: list[float] = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = time.perf_counter()
            model_fn(input_tensor)
            latencies.append((time.perf_counter() - start) * 1000)  # ms

    latencies_arr = np.array(latencies)
    result = {
        "label": label,
        "p50_ms": round(float(np.percentile(latencies_arr, 50)), 3),
        "p95_ms": round(float(np.percentile(latencies_arr, 95)), 3),
        "p99_ms": round(float(np.percentile(latencies_arr, 99)), 3),
        "mean_ms": round(float(np.mean(latencies_arr)), 3),
    }
    print(f"[{label}] p50={result['p50_ms']}ms | p95={result['p95_ms']}ms | p99={result['p99_ms']}ms")
    return result

# Establish baseline
baseline_results = benchmark_latency(model, sample_input, label="baseline_pytorch")

## 2. ONNX Export & Runtime

ONNX (Open Neural Network Exchange) is a portable format that enables hardware-specific optimizations. ONNX Runtime typically delivers 1.5-3x speedups over vanilla PyTorch for CPU inference.

In [ ]:
# Export to ONNX and benchmark
from pathlib import Path

onnx_path = Path("model_optimized.onnx")

# Export
torch.onnx.export(
    model,
    sample_input,
    str(onnx_path),
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=17,
)
print(f"ONNX model exported: {onnx_path} ({onnx_path.stat().st_size / 1024:.1f} KB)")

try:
    import onnxruntime as ort

    session = ort.InferenceSession(
        str(onnx_path),
        providers=["CPUExecutionProvider"]
    )

    input_np = sample_input.numpy()
    input_name = session.get_inputs()[0].name

    def onnx_infer(x: np.ndarray) -> np.ndarray:
        return session.run(None, {input_name: x})[0]

    # Benchmark ONNX
    onnx_latencies: list[float] = []
    for _ in range(50): onnx_infer(input_np)  # warmup
    for _ in range(500):
        t0 = time.perf_counter()
        onnx_infer(input_np)
        onnx_latencies.append((time.perf_counter() - t0) * 1000)

    onnx_p99 = float(np.percentile(onnx_latencies, 99))
    print(f"[onnx_runtime] p50={np.percentile(onnx_latencies,50):.3f}ms | p99={onnx_p99:.3f}ms")
    print(f"Speedup vs baseline: {baseline_results['p99_ms'] / onnx_p99:.2f}x")
except ImportError:
    print("onnxruntime not installed. pip install onnxruntime")

## 3. Quantization (INT8)

Quantization reduces model weights from FP32 to INT8, shrinking model size by 4x and speeding up inference — with minimal accuracy loss for most models.

In [ ]:
# Dynamic quantization: no calibration data needed
from torch.quantization import quantize_dynamic

# Apply dynamic quantization to Linear layers
quantized_model = quantize_dynamic(
    model,
    qconfig_spec={nn.Linear},
    dtype=torch.qint8
)
quantized_model.eval()

# Compare model sizes
import io
def model_size_mb(m: nn.Module) -> float:
    """Calculate model size by serializing to buffer."""
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.tell() / (1024 * 1024)

orig_size = model_size_mb(model)
quant_size = model_size_mb(quantized_model)
print(f"Original model size: {orig_size:.2f} MB")
print(f"Quantized model size: {quant_size:.2f} MB")
print(f"Size reduction: {orig_size / quant_size:.2f}x")

# Verify outputs are close
with torch.no_grad():
    orig_out = model(sample_input)
    quant_out = quantized_model(sample_input)
    diff = (orig_out - quant_out).abs().max().item()
    print(f"Max output difference (original vs quantized): {diff:.6f}")

# Benchmark quantized model
quant_results = benchmark_latency(quantized_model, sample_input, label="int8_quantized")

## 4. Dynamic Batching & Summary

Dynamic batching groups individual requests into batches to maximize GPU/CPU utilization. For CPU inference, batches of 16-64 often give the best throughput.

In [ ]:
# Dynamic batching: throughput vs latency tradeoff
import pandas as pd

results_rows = []
for batch_size in [1, 4, 8, 16, 32, 64]:
    batch_tensor = torch.randn(batch_size, 128)
    latencies: list[float] = []

    with torch.no_grad():
        for _ in range(20): model(batch_tensor)  # warmup
        for _ in range(200):
            t0 = time.perf_counter()
            model(batch_tensor)
            latencies.append((time.perf_counter() - t0) * 1000)

    total_ms = np.mean(latencies)
    per_sample_ms = total_ms / batch_size
    throughput = batch_size / (total_ms / 1000)  # samples per second
    results_rows.append({
        "batch_size": batch_size,
        "batch_latency_ms": round(total_ms, 2),
        "per_sample_ms": round(per_sample_ms, 3),
        "throughput_rps": round(throughput, 0)
    })

df = pd.DataFrame(results_rows)
print("=== Dynamic Batching: Latency vs Throughput Tradeoff ===")
print(df.to_string(index=False))
print()
best_throughput = df.loc[df["throughput_rps"].idxmax()]
print(f"Best throughput at batch_size={int(best_throughput['batch_size'])}: {best_throughput['throughput_rps']:.0f} req/s")

## Exercises

1. **TorchScript**: Compile the model with `torch.jit.script(model)` and `torch.jit.trace(model, sample_input)`. Benchmark both against the baseline. Measure the p99 latency improvement. When would you prefer script vs trace?

2. **FP16 inference**: Convert the model to half precision with `model.half()` and benchmark on CPU. Then repeat on a GPU (if available) with `model.cuda().half()`. Compare speedup on CPU vs GPU — why is the difference dramatic?

3. **Request batching queue**: Implement a `BatchingQueue` class that collects individual inference requests for up to 10ms, then runs them as a single batch. Use `threading.Timer` and a `queue.Queue`. Measure the effective throughput improvement versus sequential single-sample inference.